#Notebook Overview
This notebook is used to download spatial data and create an AnnData object for spatial transcriptomics analysis.

Current samples:
- BEME_346
- BEME_355G

Both samples represent peritoneal endometriosis lesions included in v1.

In [10]:
# -- Installs
!pip install -q scanpy\
pandas \
numpy==2.0.2 \
squidpy \
pillow \
requests \
anndata \
session-info2

In [11]:
# -- Load libraries
import gzip
import json
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import scanpy as sc
import squidpy as sq

from PIL import Image
from matplotlib import pyplot as plt
from session_info2 import session_info

import anndata as ad
ad.settings.allow_write_nullable_strings = True

In [12]:
# -- Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


In [13]:
# -- Paths
project_path = Path(
    "/content/drive/MyDrive/endo-immune-atlas"
)

raw_spatial_path = (
    project_path
    / "data"
    / "raw"
    / "spatial"
)

interim_spatial_path = (
    project_path
    / "data"
    / "interim"
    / "spatial"
)

spatial_figures_path = (
    project_path
    / "figures"
    / "spatial"
    / "data_collection"
)

raw_spatial_path.mkdir(
    parents=True,
    exist_ok=True,
)

interim_spatial_path.mkdir(
    parents=True,
    exist_ok=True,
)

spatial_figures_path.mkdir(
    parents=True,
    exist_ok=True,
)

In [14]:
# -- Files to be downloaded -- ADD NEW FILES HERE
GSM6690475_files = [
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690475/suppl/GSM6690475_BEME_346_barcodes.tsv.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690475/suppl/GSM6690475_BEME_346_features.tsv.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690475/suppl/GSM6690475_BEME_346_matrix.mtx.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690475/suppl/GSM6690475_BEME_346_scalefactors_json.json.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690475/suppl/GSM6690475_BEME_346_tissue_hires_image.png.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690475/suppl/GSM6690475_BEME_346_tissue_positions_list.csv.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690475/suppl/GSM6690475_D_V11F09-023_BEME346.tif.gz"]


GSM6690476_files = [
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690476/suppl/GSM6690476_BEME_355G_barcodes.tsv.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690476/suppl/GSM6690476_BEME_355G_features.tsv.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690476/suppl/GSM6690476_BEME_355G_matrix.mtx.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690476/suppl/GSM6690476_BEME-355G_scalefactors_json.json.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690476/suppl/GSM6690476_BEME-355G_tissue_hires_image.png.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690476/suppl/GSM6690476_BEME-355G_tissue_positions_list.csv.gz",
    "https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM6690nnn/GSM6690476/suppl/GSM6690476_C_V11F09-023_BEME355G.tif.gz"
]

sample_files = {
    "GSM6690475": GSM6690475_files,
    "GSM6690476": GSM6690476_files,
}

In [19]:
# -- DOWNLOAD FILES
for sample, files in sample_files.items():

    sample_dir = (
        raw_spatial_path
        / sample
    )

    sample_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    print(f"\nDownloading files for {sample}")

    for url in files:

        filename = (
            sample_dir
            /Path(url).name
        )

        if filename.exists():
          print(f"Skipping existing file: {filename.name}")
          continue

        print(f"Downloading {filename}")

        response = requests.get(
            url,
            stream=True,
            timeout=120,
            )


        response.raise_for_status()

        with filename.open("wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
              if chunk:
                file.write(chunk)

In [25]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNCTION 00_1: BUILD VISIUM STRUCTURE FROM DOWNLOADED FILES
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def prepare_visium_directory(gsm_dir):

  """
  Build the directory structure expected for a Visium sample.

  Creates:
  -filtered_feature_bc_matrix/
  -spatial/
  """

  gsm_dir = Path(gsm_dir)

  matrix_dir = gsm_dir / "filtered_feature_bc_matrix"

  spatial_dir = gsm_dir / "spatial"

  matrix_dir.mkdir(exist_ok=True, parents=True)
  spatial_dir.mkdir(exist_ok=True, parents=True)

  matrix_patterns = {
      "*barcodes.tsv.gz": (
          matrix_dir
          / "barcodes.tsv.gz"
          ),
      "*features.tsv.gz": (
          matrix_dir
          / "features.tsv.gz"
          ),
      "*matrix.mtx.gz": (
          matrix_dir
          / "matrix.mtx.gz"
          )
      }

  spatial_patterns = {
      "*scalefactors_json.json.gz": (
          spatial_dir
          / "scalefactors_json.json"
          ),
      "*tissue_hires_image.png.gz": (
          spatial_dir
          / "tissue_hires_image.png"
          ),
      "*tissue_positions_list.csv.gz": (
          spatial_dir
          / "tissue_positions_list.csv"
          )
      }

  # -- Copy matrix files
  for pattern, destination in matrix_patterns.items():

    matches = list(gsm_dir.glob(pattern))

    if not matches:
      print(f"Missing: {pattern}")
      continue

    if destination.exists():
      print(f"Skipping existing file: "
            f"{destination.name}")
      continue


    shutil.copy2(
        matches[0],
        destination,
    )


    # -- Decompress

  for pattern, destination in spatial_patterns.items():

    matches = list(gsm_dir.glob(pattern))

    if not matches:
      print(f"Missing: {pattern}")
      continue

    if destination.exists():
      print(
          f"Skipping existing file:"
          f"{destination.name}"
      )
      continue

    with gzip.open(
        matches[0],
        "rb",
    ) as input_file:
      with destination.open(
          "wb") as output_file:

          shutil.copyfileobj(
          input_file,
          output_file,)

In [26]:
# -- Prepare Visium directories
prepare_visium_directory(
    raw_spatial_path
    / "GSM6690475"
)

prepare_visium_directory(
    raw_spatial_path
    / "GSM6690476"
)

In [28]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNCTION 00_2: CREATE A VISIUM OBJECT
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def create_visium_object(gsm_dir, library_id, gsm_id):
  """
  Create a Visium-compatible AnnData object from a prepared directory.
  """
  gsm_dir = Path(gsm_dir)

  # - Load counts
  adata = sc.read_10x_mtx(
      gsm_dir
      / "filtered_feature_bc_matrix",
      var_names="gene_symbols",
      make_unique=True
  )

  # - Load tissue coordinates
  coords = pd.read_csv(
      gsm_dir
      / "spatial"
      / "tissue_positions_list.csv",
      header=None
  )

  coords.columns = [
      "barcode",
      "in_tissue",
      "array_row",
      "array_col",
      "pxl_row",
      "pxl_col"
  ]

  coords = coords.set_index("barcode")


  missing_barcodes = (
      adata.obs_names.difference(coords.index)
  )

  if len(missing_barcodes) > 0:
    raise ValueError(
        f"{len(missing_barcodes)} barcodes are missing spatial coords..."
    )

  # - Map coodinates to tissues
  coords = coords.loc[adata.obs_names]

  adata.obs["in_tissue"] = coords["in_tissue"].astype(int).values


  adata.obs["array_row"] = coords["array_row"].astype(int).values

  adata.obs["array_col"] = coords["array_col"].astype(int).values

  adata.obsm["spatial"] = coords[["pxl_col", "pxl_row"]].to_numpy()

  # - Load scaling factors
  with (gsm_dir
        / "spatial"
        / "scalefactors_json.json").open() as file:

        scalefactors = json.load(file)


  # - Load tissue histology image
  hires_img = np.array(
      Image.open(
          gsm_dir
          / "spatial"
          / "tissue_hires_image.png"
      )
  )

  # - Load spatial information
  adata.uns["spatial"] = {
      library_id: {
      "images": {
          "hires":hires_img
      },
      "scalefactors": scalefactors,
      "metadata": {
          "library_id": library_id,
          "gsm_id": gsm_id
      }
  }
}

  # - Add sample metadata
  adata.obs["library_id"] = library_id
  adata.obs["sample_id"] = library_id
  adata.obs["gsm_id"] = gsm_id
  adata.obs["tissue_type"] = "EcP"
  adata.obs["condition"] = "endometriosis"
  adata.obs["lesion_site"] = "peritoneal"

  return adata

In [29]:
# -- Load Visium samples as AnnData objects
adata_346 = create_visium_object(
    gsm_dir=(
        raw_spatial_path
        / "GSM6690475"
    ),
    library_id="BEME_346",
    gsm_id="GSM6690475",
)

adata_355G = create_visium_object(
    gsm_dir=(
        raw_spatial_path
        / "GSM6690476"
    ),
    library_id="BEME_355G",
    gsm_id="GSM6690476",
)

In [42]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# FUNCTION 00_3: OBJECT VALIDATION
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
def validate_visium_object(adata, library_id):
  """
  Print object information and save two spatial validation plots.
  """
  print(f"\n{library_id}")
  print(adata)

  print(f"The shape of data is: {adata.obsm['spatial'].shape}")

  print("In-tissue values:")


  print(adata.obs["in_tissue"]
        .value_counts()
        .sort_index())

  # - Histology image check
  sq.pl.spatial_scatter(adata,
                        color=None,
                        library_id=library_id,
                        )

  plt.savefig(
      spatial_figures_path
      / f"00_{library_id}_histology_check.png",
      bbox_inches="tight",
      dpi=300,
  )

  plt.close()


  # In tissue spot check
  sq.pl.spatial_scatter(adata,
                        color="in_tissue",
                        library_id = library_id,
                        )

  plt.savefig(
        spatial_figures_path
        / f"00_{library_id}_in_tissue_check.png",
        bbox_inches="tight",
        dpi=300,
    )


  plt.close()

In [43]:
# -- Validate both objects
validate_visium_object(
    adata_346,
    library_id="BEME_346",
)

validate_visium_object(
    adata_355G,
    library_id="BEME_355G",
)


BEME_346
AnnData object with n_obs × n_vars = 1388 × 36601
    obs: 'in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site'
    var: 'gene_ids', 'feature_types'
    uns: 'spatial'
    obsm: 'spatial'
The shape of data is: (1388, 2)
In-tissue values:
in_tissue
1    1388
Name: count, dtype: int64

BEME_355G
AnnData object with n_obs × n_vars = 1960 × 36601
    obs: 'in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site'
    var: 'gene_ids', 'feature_types'
    uns: 'spatial'
    obsm: 'spatial'
The shape of data is: (1960, 2)
In-tissue values:
in_tissue
1    1960
Name: count, dtype: int64


In [44]:
# ==========================================================================
# FUNCTION 00_4: SAVE OBJECT
# ==========================================================================

def save_object(
    adata,
    output_path,
    library_id,
):
    """
    Save and validate a spatial AnnData object.
    """

    output_path = Path(
        output_path
    )

    output_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    file_path = (
        output_path
        / f"{library_id}_raw.h5ad"
    )

    adata.write_h5ad(
        file_path
    )

    # -- Validate without loading the full object into memory
    saved_object = sc.read_h5ad(
        file_path,
        backed="r",
    )

    print(
        f"Saved {library_id}: "
        f"{saved_object.shape}"
    )

    print(
        f"obs columns: "
        f"{saved_object.obs.columns.tolist()}"
    )

    saved_object.file.close()

In [45]:
# -- Save objects
save_object(
    adata_346,
    interim_spatial_path,
    library_id="BEME_346",
)

save_object(
    adata_355G,
    interim_spatial_path,
    library_id="BEME_355G",
)

Saved BEME_346: (1388, 36601)
obs columns: ['in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site']
Saved BEME_355G: (1960, 36601)
obs columns: ['in_tissue', 'array_row', 'array_col', 'library_id', 'sample_id', 'gsm_id', 'tissue_type', 'condition', 'lesion_site']


In [46]:
## -- Run to see session info
session_info()

AttributeError: 'HBox' object has no attribute '_repr_mimebundle_'

numpy	2.0.2
pandas	2.3.3 (3.0.5)
requests	2.32.4
scanpy	1.12.3
squidpy	1.8.3
pillow	11.3.0
matplotlib	3.10.0
anndata	0.12.19
google-colab	1.0.0
----	----
Python	3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OS	Linux-6.6.122+-x86_64-with-glibc2.35
Updated	2026-08-03 03:43